# Flipkart Traffic Demand Prediction Pipeline

This notebook executes the end-to-end spatial-temporal regression pipeline for the Flipkart Grid challenge. It features strict Out-Of-Fold target encoding, momentum tracking, and Optuna-optimized LightGBM predictions.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.metrics import r2_score
import lightgbm as lgb
import optuna
import os

warnings.filterwarnings('ignore')
print("✅ Libraries imported successfully.")

## 1. Load Data
Load `train.csv` and `test.csv`. Ensure the dataset is located in the `dataset/` directory.

In [ ]:
dataset_dir = os.path.join(os.path.dirname(os.path.abspath('')), 'dataset')

print(f"Loading data from: {dataset_dir}...")
train = pd.read_csv(os.path.join(dataset_dir, 'train.csv'))
test = pd.read_csv(os.path.join(dataset_dir, 'test.csv'))
test_indices = test['Index'].values

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

## 2. Base Feature Engineering
Extract temporal components, cyclical time representations, and impute missing weather information. We also create hierarchy mapping for geohashes.

In [ ]:
print("Engineering base temporal and spatial features...")

for df in [train, test]:
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather'] = df['Weather'].fillna('Unknown')
    df['NumberofLanes'] = df['NumberofLanes'].fillna(0).astype(int)
    df['LargeVehicles'] = df['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(0).astype(int)
    df['Landmarks'] = df['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
    
    time_split = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = time_split[0]
    df['minute'] = time_split[1]
    df['slot'] = (df['hour'] * 60 + df['minute']) // 15
    
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['is_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 10) | (df['hour'] >= 16) & (df['hour'] <= 19)).astype(int)

# fill missing temps per weather condition, fallback to global median
weather_medians = train.groupby('Weather')['Temperature'].median()
train['Temperature'] = train.groupby('Weather')['Temperature'].transform(lambda x: x.fillna(x.median()))
train['Temperature'] = train['Temperature'].fillna(train['Temperature'].median())
test['Temperature'] = test['Weather'].map(weather_medians).fillna(train['Temperature'].median())

weather_map = {'Sunny': 0, 'Foggy': 1, 'Rainy': 2, 'Snowy': 3, 'Unknown': 0}
train['weather_sev'] = train['Weather'].map(weather_map)
test['weather_sev'] = test['Weather'].map(weather_map)

for df in [train, test]:
    df['gh5'] = df['geohash'].str[:5]
    df['gh4'] = df['geohash'].str[:4]
    
print("✅ Base features generated.")

## 3. Strict Out-Of-Fold (OOF) Target Encoding
To prevent target leakage, we implemented a strict time-based split on Day 48. We also trained on slots 0-47 and validated on slots 48-95.

In [ ]:
print("Setting up strict chron split for cross-validation on Day 48...")

day48 = train[train['day'] == 48].copy()
train_fold = day48[day48['slot'] <= 47].copy()
val_fold = day48[day48['slot'] > 47].copy()

d48_morning = train_fold[train_fold['slot'] <= 8].copy()
morning_stats = d48_morning.groupby('geohash')['demand'].agg(
    morning_mean='mean'
).reset_index()
last_obs = d48_morning.sort_values('slot').groupby('geohash').last().reset_index()[['geohash', 'demand']].rename(columns={'demand': 'lag1'})
morning_stats = morning_stats.merge(last_obs, on='geohash', how='left')

train_fold = train_fold.merge(morning_stats, on='geohash', how='left')
val_fold = val_fold.merge(morning_stats, on='geohash', how='left')

global_morning_mean = d48_morning['demand'].mean()
for df in [train_fold, val_fold]:
    df['morning_mean'] = df['morning_mean'].fillna(global_morning_mean)
    df['lag1'] = df['lag1'].fillna(df['morning_mean'])

print("✅ Morning momentum stats attached for CV.")

## 4. OOF Spatial & Momentum Encodings
We compute the mean demand for exact geohashes, parent geohashes, and shift ratios (momentum).

In [ ]:
print("Computing OOF spatial encodings...")

ghh_mean_cv = train_fold.groupby(['geohash', 'hour'])['demand'].mean()
global_mean_cv = train_fold['demand'].mean()

train_fold['ghh_encoded'] = train_fold.set_index(['geohash', 'hour']).index.map(ghh_mean_cv).fillna(global_mean_cv)
val_fold['ghh_encoded'] = val_fold.set_index(['geohash', 'hour']).index.map(ghh_mean_cv).fillna(global_mean_cv)

city_hour_mean_cv = train_fold.groupby('hour')['demand'].mean()
train_fold['city_hour_momentum'] = train_fold['hour'].map(city_hour_mean_cv).fillna(global_mean_cv)
val_fold['city_hour_momentum'] = val_fold['hour'].map(city_hour_mean_cv).fillna(global_mean_cv)

d48_morning_global = train_fold[train_fold['slot'] <= 8].groupby('geohash')['demand'].mean().reset_index()
d48_morning_global = d48_morning_global.rename(columns={'demand': 'd48_morning_mean'})

train_fold = train_fold.merge(d48_morning_global, on='geohash', how='left')
val_fold = val_fold.merge(d48_morning_global, on='geohash', how='left')

train_fold['morning_shift_ratio'] = (train_fold['morning_mean'] / (train_fold['d48_morning_mean'] + 1e-9)).fillna(1.0)
val_fold['morning_shift_ratio'] = (val_fold['morning_mean'] / (val_fold['d48_morning_mean'] + 1e-9)).fillna(1.0)

train_fold['temp_x_rush'] = train_fold['Temperature'] * train_fold['is_rush']
val_fold['temp_x_rush'] = val_fold['Temperature'] * val_fold['is_rush']

temp_bins = pd.qcut(train_fold['Temperature'], q=5, labels=False, duplicates='drop').unique()
train_fold['temp_bin'] = pd.qcut(train_fold['Temperature'], q=5, labels=False, duplicates='drop')
val_fold['temp_bin'] = pd.cut(val_fold['Temperature'], bins=pd.qcut(train_fold['Temperature'], q=5, retbins=True)[1], labels=False, include_lowest=True).fillna(0).astype(int)

train_fold['temp_bin_x_hour'] = train_fold['temp_bin'] * train_fold['hour']
val_fold['temp_bin_x_hour'] = val_fold['temp_bin'] * val_fold['hour']

# computed on train_fold only to avoid leakage
gh5h_mean_cv = train_fold.groupby(['gh5', 'hour'])['demand'].mean()
gh4h_mean_cv = train_fold.groupby(['gh4', 'hour'])['demand'].mean()

train_fold['gh5h_encoded'] = train_fold.set_index(['gh5', 'hour']).index.map(gh5h_mean_cv).fillna(global_mean_cv)
val_fold['gh5h_encoded'] = val_fold.set_index(['gh5', 'hour']).index.map(gh5h_mean_cv).fillna(global_mean_cv)
train_fold['gh4h_encoded'] = train_fold.set_index(['gh4', 'hour']).index.map(gh4h_mean_cv).fillna(global_mean_cv)
val_fold['gh4h_encoded'] = val_fold.set_index(['gh4', 'hour']).index.map(gh4h_mean_cv).fillna(global_mean_cv)

print("✅ Encodings completed.")

## 5. Model Validation (CV)
Trained an Optuna-optimized LightGBM model on the training fold and evaluate on the validation fold.

In [ ]:
features = [
    'hour', 'minute', 'slot', 'hour_sin', 'hour_cos', 'is_rush',
    'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature',
    'ghh_encoded', 'morning_mean', 'lag1', 'morning_shift_ratio',
    'temp_x_rush', 'temp_bin_x_hour',
    'gh5h_encoded', 'gh4h_encoded'
]
cat_features = ['geohash', 'RoadType', 'Weather']

X_tr = train_fold[features + cat_features]
y_tr = train_fold['demand']
X_val = val_fold[features + cat_features]
y_val = val_fold['demand']

for col in cat_features:
    X_tr[col] = X_tr[col].astype('category')
    X_val[col] = X_val[col].astype('category')

# optuna-tuned 
lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.04,
    'num_leaves': 54,
    'max_depth': 7,
    'min_child_samples': 12,
    'reg_lambda': 0.15609878207944705,
    'reg_alpha': 0.02009412265635894,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print(f"Training LightGBM on CV split with {len(features)} features...")
model = lgb.LGBMRegressor(**lgb_params)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_features,
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

val_preds = np.clip(model.predict(X_val), 0, 1)
cv_r2 = r2_score(y_val, val_preds)
print(f"🎯 Validation R2 Score: {cv_r2:.5f}")

## 6. Final Data Preparation
To build the final model for the leaderboard, we refit on all available data (Day 48 completely + Day 49 morning) and recompute all encodings globally.

In [ ]:
print("Preparing data for final model fit...")
# final fit on all available data
final_train = train[(train['day'] == 48) | ((train['day'] == 49) & (train['slot'] <= 8))].copy()

d48_morn_train = train[(train['day'] == 48) & (train['slot'] <= 8)]
d48_morning_stats = d48_morn_train.groupby('geohash')['demand'].agg(
    morning_mean='mean'
).reset_index()
d48_last_obs = d48_morn_train.sort_values('slot').groupby('geohash').last().reset_index()[['geohash', 'demand']].rename(columns={'demand': 'lag1'})
d48_morning_stats = d48_morning_stats.merge(d48_last_obs, on='geohash', how='left')
d48_morning_stats['day'] = 48

d49_morn_train = train[(train['day'] == 49) & (train['slot'] <= 8)]
d49_morning_stats = d49_morn_train.groupby('geohash')['demand'].agg(
    morning_mean='mean'
).reset_index()
d49_last_obs = d49_morn_train.sort_values('slot').groupby('geohash').last().reset_index()[['geohash', 'demand']].rename(columns={'demand': 'lag1'})
d49_morning_stats = d49_morning_stats.merge(d49_last_obs, on='geohash', how='left')
d49_morning_stats['day'] = 49

all_morning_stats = pd.concat([d48_morning_stats, d49_morning_stats])
final_train = final_train.merge(all_morning_stats, on=['geohash', 'day'], how='left')

test = test.merge(d49_morning_stats[['geohash', 'morning_mean', 'lag1']], on='geohash', how='left')

global_morning_mean_final = train[train['slot'] <= 8]['demand'].mean()
for df in [final_train, test]:
    df['morning_mean'] = df['morning_mean'].fillna(global_morning_mean_final)
    df['lag1'] = df['lag1'].fillna(df['morning_mean'])

d48_hour_mean = train[train['day'] == 48].groupby(['geohash', 'hour'])['demand'].mean()
d48_global_mean = train[train['day'] == 48]['demand'].mean()

final_train['ghh_encoded'] = final_train.set_index(['geohash', 'hour']).index.map(d48_hour_mean).fillna(d48_global_mean)
test['ghh_encoded'] = test.set_index(['geohash', 'hour']).index.map(d48_hour_mean).fillna(d48_global_mean)

final_city_hour_mean = final_train.groupby('hour')['demand'].mean()
final_global_mean = final_train['demand'].mean()
final_train['city_hour_momentum'] = final_train['hour'].map(final_city_hour_mean).fillna(final_global_mean)
test['city_hour_momentum'] = test['hour'].map(final_city_hour_mean).fillna(final_global_mean)

d48_morning_global_final = train[(train['day'] == 48) & (train['slot'] <= 8)].groupby('geohash')['demand'].mean().reset_index()
d48_morning_global_final = d48_morning_global_final.rename(columns={'demand': 'd48_morning_mean'})

final_train = final_train.merge(d48_morning_global_final, on='geohash', how='left')
final_train['morning_shift_ratio'] = np.where(final_train['day'] == 49, final_train['morning_mean'] / (final_train['d48_morning_mean'] + 1e-9), 1.0)
final_train['morning_shift_ratio'] = final_train['morning_shift_ratio'].fillna(1.0)

test = test.merge(d48_morning_global_final, on='geohash', how='left')
test['morning_shift_ratio'] = (test['morning_mean'] / (test['d48_morning_mean'] + 1e-9)).fillna(1.0)

final_train['temp_x_rush'] = final_train['Temperature'] * final_train['is_rush']
test['temp_x_rush'] = test['Temperature'] * test['is_rush']

final_train['temp_bin'] = pd.qcut(final_train['Temperature'], q=5, labels=False, duplicates='drop')
test['temp_bin'] = pd.cut(test['Temperature'], bins=pd.qcut(final_train['Temperature'], q=5, retbins=True)[1], labels=False, include_lowest=True).fillna(0).astype(int)

final_train['temp_bin_x_hour'] = final_train['temp_bin'] * final_train['hour']
test['temp_bin_x_hour'] = test['temp_bin'] * test['hour']

gh5h_mean_final = train[train['day'] == 48].groupby(['gh5', 'hour'])['demand'].mean()
gh4h_mean_final = train[train['day'] == 48].groupby(['gh4', 'hour'])['demand'].mean()

final_train['gh5h_encoded'] = final_train.set_index(['gh5', 'hour']).index.map(gh5h_mean_final).fillna(d48_global_mean)
test['gh5h_encoded'] = test.set_index(['gh5', 'hour']).index.map(gh5h_mean_final).fillna(d48_global_mean)
final_train['gh4h_encoded'] = final_train.set_index(['gh4', 'hour']).index.map(gh4h_mean_final).fillna(d48_global_mean)
test['gh4h_encoded'] = test.set_index(['gh4', 'hour']).index.map(gh4h_mean_final).fillna(d48_global_mean)

print("✅ Final encodings completed on all available data.")

## 7. Prediction & Submission
We fit the final model to our newly constructed global dataset and generate the target leaderboard predictions.

In [ ]:
X_final = final_train[features + cat_features].copy()
y_final = final_train['demand']
X_test = test[features + cat_features].copy()

for col in cat_features:
    X_final[col] = X_final[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f"Training final model on {X_final.shape[0]} rows...")
model.fit(X_final, y_final, categorical_feature=cat_features)

print("Generating predictions...")
final_preds = np.clip(model.predict(X_test), 0, 1)

submission = pd.DataFrame({'Index': test_indices, 'demand': final_preds})
submission.to_csv('submission.csv', index=False)

print(f"📊 Final Submission Mean Demand: {submission['demand'].mean():.5f}")
print("✅ Saved: submission.csv - Ready for upload!")